In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from Base_Modules.Environments import Prison
from Prison_Strategies.Basic_Strategies import *
from Base_Modules.Game_Master import Game_Master
from Base_Modules.Action import Action_History, Prison_Actions, Duel_Matrix
import pandas as pd
from collections import defaultdict

In [4]:
prison = Prison()
actions = prison.Get_Actions()

In [5]:
strategies_list = [
    Random_Strategy(actions=actions),
    Random_Strategy(actions=actions, p_coop=0.1),
    Always_Betray(actions=actions),
    Always_Cooperate(actions=actions),
    Patient_Unforgiving(actions=actions),
    Copycat(actions=actions),
    Periodic(actions=actions, period=1),
]

number_of_strategies = len(strategies_list)

In [6]:
def Get_Index_By_Name(strategies : dict[int, Strategy], name : str) -> int:
    for ix, (id, s) in enumerate(strategies.items()):
        if str(s) == name:
            return id
    for ix, (id, s) in enumerate(strategies.items()):
        if str(s).startswith(name):
            return id
    return -1

In [7]:
strategies = {}

for (i, s) in enumerate(strategies_list):
    strategies[i] = s
    s.Set_ID(i)

In [8]:
number_of_games = 10
total_games_explicit = True
max_action_memory = -1

gm = Game_Master(prison, strategies=strategies, duel_size=2, max_action_memory=max_action_memory)
duel_matrix, rewards = gm.Tournament(number_of_games, Game_Master.Game_Type.All_Vs_All, total_games_explicit=total_games_explicit)
rewards.Sort_Total_Rewards()

In [9]:
print(duel_matrix.Get_Action_History((0, 1)).Get_Action_History())
print(Get_Index_By_Name(strategies, "Periodic"))

{0: [<Prison_Actions.Betray: 1>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Betray: 1>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Cooperate: 0>, <Prison_Actions.Betray: 1>], 1: [<Prison_Actions.Betray: 1>, <Prison_Actions.Betray: 1>, <Prison_Actions.Betray: 1>, <Prison_Actions.Betray: 1>, <Prison_Actions.Betray: 1>, <Prison_Actions.Betray: 1>, <Prison_Actions.Betray: 1>, <Prison_Actions.Betray: 1>, <Prison_Actions.Betray: 1>, <Prison_Actions.Betray: 1>]}
6


In [10]:
total_rewards = rewards.Get_Total_Rewards()
total_rewards_per_name = {str(strategies[i]):total_rewards[i] for i in total_rewards.keys()}

average_rewards_per_match = {k: (float(v)/number_of_strategies) for k, v in total_rewards_per_name.items()}
average_rewards_per_round = {k: (float(v)/(number_of_strategies*number_of_games)) for k, v in total_rewards_per_name.items()}

duel_rewards = rewards.Get_All_Duel_Rewards()
winner = next(iter(total_rewards))

## Wyniki

In [11]:
average_reward_per_round_df = pd.DataFrame.from_dict(average_rewards_per_round, orient="index", columns=["Average Reward"])
average_reward_per_round_df.index.name = "Strategy Name"
average_reward_per_round_df = average_reward_per_round_df.round(3)
average_reward_per_round_df

,Average Reward
Strategy Name,
Always_Betray,2.229
Random_Strategy (p_coop=0.1),2.114
Patient_Unforgiving (patience=1),1.971
Periodic (period=1),1.814
Copycat (1st:Cooperate),1.771
Always_Cooperate,1.457
Random_Strategy (p_coop=0.5),1.343


In [ ]:


def _nemesis_worst_score(id : int, enemy_id : int, nemesis : tuple[int, dict[int, int]], new_result : dict[int, int]) -> int:
    if new_result[id] < nemesis[1][id]:
        return (enemy_id, new_result)
    return nemesis

def _nemesis_largest_difference(id)

def Get_Nemesis(duel_rewards, criterion) -> dict:
    strategy_nemesis = {}
    for (duel_ids, results) in duel_rewards.items():
        for id in duel_ids:
            enemy_id = ([i for i in duel_ids if i != id])[0]
            if strategy_nemesis.get(id) == None:
                strategy_nemesis[id] = (enemy_id, {id : float("inf"), enemy_id : -float("inf")})
            strategy_nemesis[id] = criterion(id, enemy_id, strategy_nemesis[id], results)
    return strategy_nemesis

print(Get_Nemesis(duel_rewards=duel_rewards, criterion=_nemesis_worst_score)[1])


(2, {1: 7, 2: 22})


In [ ]:
# Per match

average_reward_per_match_df = pd.DataFrame.from_dict(average_rewards_per_match, orient="index", columns=["Average Reward"])
average_reward_per_match_df.index.name = "Strategy Name"
average_reward_per_match_df = average_reward_per_match_df.round(3)
# average_reward_per_match_df

In [ ]:
strat_names = [str(s) for s in strategies.values()]

score_matrix = pd.DataFrame(index=strat_names, columns=strat_names, dtype=object)

for (s1, s2), scores in duel_rewards.items():
    score_matrix.loc[str(strategies[s2]), str(strategies[s1])] = (scores[s2], scores[s1])
    score_matrix.loc[str(strategies[s1]), str(strategies[s2])] = (scores[s1], scores[s2])

for s in strat_names:
    score_matrix.loc[s, s] = (0, 0)

In [ ]:
victory_matrix = score_matrix.apply(lambda col: col.map(lambda x: int(x[0] > x[1])))
for s in strat_names:
    victory_matrix.loc[s, s] = float("NaN")

In [ ]:
def color_cell(x):
    if not isinstance(x, tuple):
        return ""
    if x[0] == 0 and x[1] == 0:
        return "background-color: black"
    elif x[0] > x[1]:
        return "background-color: green"
    elif x[0] < x[1]:
        return "background-color: darkred"
    else:
        return "background-color: gray"

styled_score_matrix = (
    score_matrix.style
    .map(color_cell)
    .set_properties(**{
        "border": "1px solid black"
    })
    .set_table_styles([
        {"selector": "th", "props": [("border", "2px solid black")]},
        {"selector": "td", "props": [("border", "2px solid black")]}
    ])
)

display(styled_score_matrix)

,Random_Strategy (p_coop=0.5),Random_Strategy (p_coop=0.1),Always_Betray,Always_Cooperate,Patient_Unforgiving (patience=1),Copycat (1st:Cooperate),Periodic (period=1)
Random_Strategy (p_coop=0.5),"(0, 0)","(3, 38)","(7, 22)","(42, 12)","(10, 25)","(27, 27)","(22, 22)"
Random_Strategy (p_coop=0.1),"(38, 3)","(0, 0)","(9, 14)","(50, 0)","(15, 15)","(19, 14)","(27, 12)"
Always_Betray,"(22, 7)","(14, 9)","(0, 0)","(50, 0)","(14, 9)","(14, 9)","(30, 5)"
Always_Cooperate,"(12, 42)","(0, 50)","(0, 50)","(0, 0)","(30, 30)","(30, 30)","(15, 40)"
Patient_Unforgiving (patience=1),"(25, 10)","(15, 15)","(9, 14)","(30, 30)","(0, 0)","(30, 30)","(27, 12)"
Copycat (1st:Cooperate),"(27, 27)","(14, 19)","(9, 14)","(30, 30)","(30, 30)","(0, 0)","(23, 28)"
Periodic (period=1),"(22, 22)","(12, 27)","(5, 30)","(40, 15)","(12, 27)","(28, 23)","(0, 0)"


In [ ]:
import webbrowser
store_styled_matrix_in_html = False
if store_styled_matrix_in_html:
    styled_score_matrix.to_html("styled_score_matrix.html")
    webbrowser.open_new_tab("styled_score_matrix.html")